<a href="https://colab.research.google.com/github/SaadAh-mad/speaker-verification-experiments/blob/googlecolab_exp5%266/AcouspikeVoxceleb%2BWavLM%2BAAMSoftmax%2BASTP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
!pip install -q soundfile

In [16]:
#IMPORTS+CLASSES+Dataset Creation

# Install required libraries
!pip install -q transformers datasets soundfile accelerate scikit-learn

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
import os
import gc
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
from datasets import load_dataset
import datasets
import io
import soundfile as sf

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Checkpoint paths
checkpoint_dir = "/content/drive/MyDrive" if os.path.exists("/content/drive/MyDrive") else "/content"
checkpoint_path = os.path.join(checkpoint_dir, "best_voxceleb1_wavlm_astp_aam.pth")

# Split configuration: 1251 total speakers in VoxCeleb1
EVAL_SPEAKERS = set(f"id{i}" for i in range(10001, 10041))        # 40 speakers
TRAIN_SPEAKERS = sorted([f"id{i}" for i in range(10041, 11252)])  # 1211 speakers
spk2idx = {spk: idx for idx, spk in enumerate(TRAIN_SPEAKERS)}
num_speakers = len(TRAIN_SPEAKERS)

def get_speaker_id(key):
    parts = key.split("/")
    if len(parts) >= 3:
        return parts[2]
    return None

#################################################
# ASTP Pooling
#################################################
class ASTP(nn.Module):
    def __init__(self, in_dim, bottleneck_dim=128):
        super().__init__()
        self.linear1 = nn.Conv1d(in_dim, bottleneck_dim, kernel_size=1)
        self.linear2 = nn.Conv1d(bottleneck_dim, in_dim, kernel_size=1)

    def forward(self, x):
        alpha = torch.tanh(self.linear1(x))
        alpha = torch.softmax(self.linear2(alpha), dim=2)
        mean = torch.sum(alpha * x, dim=2)
        var = torch.sum(alpha * (x ** 2), dim=2) - mean ** 2
        std = torch.sqrt(var.clamp(min=1e-7))
        return torch.cat([mean, std], dim=1)

#################################################
# AAM-SoftMax
#################################################
class AAMSoftmax(nn.Module):
    def __init__(self, embedding_dim, num_classes, margin=0.2, scale=30):
        super().__init__()
        self.margin = margin
        self.scale = scale
        self.weight = nn.Parameter(torch.randn(num_classes, embedding_dim))
        nn.init.xavier_normal_(self.weight)

    def forward(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        weight = F.normalize(self.weight, dim=1)
        cosine = F.linear(embeddings, weight)

        theta = torch.acos(torch.clamp(cosine, -1 + 1e-7, 1 - 1e-7))
        target_cosine = torch.cos(theta + self.margin)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1)

        logits = (one_hot * target_cosine + (1 - one_hot) * cosine)
        logits *= self.scale
        return logits

#################################################
# Model
#################################################
class WavLMSpeakerModel(nn.Module):
    def __init__(self, num_speakers):
        super().__init__()
        self.wavlm = WavLMModel.from_pretrained("microsoft/wavlm-base")
        self.layer_weights = nn.Parameter(torch.ones(13))

        for p in self.wavlm.parameters():
            p.requires_grad = False

        self.pooling = ASTP(768)
        self.aam = AAMSoftmax(embedding_dim=1536, num_classes=num_speakers)

    def forward(self, input_values, labels=None):
        with torch.no_grad():
            outputs = self.wavlm(input_values=input_values, output_hidden_states=True)
        hidden_states = outputs.hidden_states

        weights = torch.softmax(self.layer_weights, dim=0)
        combined = 0
        for w, h in zip(weights, hidden_states):
            combined = combined + w * h

        combined = combined.transpose(1, 2)
        embedding = self.pooling(combined)
        embedding = F.normalize(embedding, dim=-1)

        logits = None
        if labels is not None:
            logits = self.aam(embedding, labels)

        return logits, embedding

# Processor and Collate Function
processor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base")

def collate_fn(batch):
    audios = [x[0] for x in batch]
    labels = torch.tensor([x[1] for x in batch])
    features = processor(
        audios,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )
    return features.input_values, labels

#################################################
# Streaming Dataset Wrapper
#################################################
class VoxCelebStreamingDataset(IterableDataset):
    def __init__(self, stream, spk2idx):
        self.stream = stream
        self.spk2idx = spk2idx

    def __iter__(self):
        for sample in self.stream:
            try:
                key = sample["__key__"]
                spk_id = get_speaker_id(key)

                if spk_id not in self.spk2idx:
                    continue

                label = self.spk2idx[spk_id]

                audio_data = sample["wav"]

                if audio_data is None:
                    continue

                audio_bytes = audio_data["bytes"]

                audio, sr = sf.read(
    io.BytesIO(audio_bytes),
    dtype="float32"
)

                if audio.ndim > 1:
                    audio = audio.mean(axis=1)

                audio = np.asarray(audio, dtype=np.float32)

                if len(audio.shape) > 1:
                    audio = audio.squeeze()

                target_len = 48000

                if len(audio) > target_len:
                    start = np.random.randint(
                    0,
                    len(audio) - target_len
                )
                    audio = audio[start:start+target_len]
                else:
                    audio = np.pad(
                    audio,
                    (0, target_len-len(audio))
                )

                yield audio, label

            except Exception as e:
                raise e


# Load & Filter Dataset
print("Loading Acouspike/Voxceleb1 dataset in streaming mode...")
hf_dataset = load_dataset("Acouspike/Voxceleb1", split="train", streaming=True)
#hf_dataset = hf_dataset.cast_column("wav", datasets.Audio(sampling_rate=16000))

# Filter out test speakers from training stream
def is_train_sample(sample):
    spk_id = get_speaker_id(sample["__key__"])
    return spk_id is not None and spk_id not in EVAL_SPEAKERS

train_stream = hf_dataset.filter(is_train_sample)
#train_stream = train_stream.shuffle(buffer_size=1000, seed=42)
dataset = VoxCelebStreamingDataset(train_stream, spk2idx)

loader = DataLoader(
    dataset,
    batch_size=8,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

Device: cuda
Loading Acouspike/Voxceleb1 dataset in streaming mode...


In [17]:
print("Testing one batch...")

for input_values, labels in loader:
    print(input_values.shape)
    print(labels.shape)
    break

Testing one batch...
torch.Size([8, 48000])
torch.Size([8])


In [18]:
model = WavLMSpeakerModel(num_speakers).to(DEVICE)

for input_values, labels in loader:
    input_values = input_values.to(DEVICE)
    labels = labels.to(DEVICE)

    logits, embeddings = model(input_values, labels)

    print("Logits shape:", logits.shape)
    print("Embeddings shape:", embeddings.shape)

    break

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

Logits shape: torch.Size([8, 1211])
Embeddings shape: torch.Size([8, 1536])


In [ ]:
print("Testing one batch...")

for input_values, labels in loader:
    print("Input shape:", input_values.shape)
    print("Labels shape:", labels.shape)
    break

Testing one batch...
<class 'dict'>
{'bytes': b'RIFF&M\x03\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00\x80>\x00\x00\x00}\x00\x00\x02\x00\x10\x00data\x02M\x03\x00\xfb\xfbM\xfc\x9c\xfc\x04\xfd\xc6\xfc\xa9\xfc\xf5\xfc\x18\xfd\xe6\xfd\x12\xfe\xa1\xfd\xb5\xfd\xaf\xfd\x0e\xfe\xe0\xfe\x1f\xff\xc6\xfe\x15\xfeu\xfd\x8e\xfd\x8f\xfe\xab\xff\xdb\xff9\xffR\xfe\x19\xfd)\xfd\x95\xfd\x9b\xfd\xd8\xfd!\xfd,\xfc\x0f\xfc\x05\xfcd\xfc\x1b\xfd$\xfd\xcc\xfc\x1f\xfdp\xfd\x1b\xfe\xf8\xfe\xfa\xfe\xee\xfe/\xff\xcc\xffa\x00$\x01v\x01.\x01i\x01\x81\x01\xd0\x01\xa7\x02\xe1\x02\xcd\x02c\x02\x00\x02\xe3\x01\x14\x02%\x02\xf4\x01\xa8\x01E\x01\xd6\x00\x93\x00(\x00)\x00\x06\x00\xb9\xff}\xff\x12\xff\xc9\xfe\x05\xff\x1e\xff\x1d\xff\x93\xffM\xffK\xff&\xff\xcf\xfe\xef\xfe\xea\xfe\t\xff\xcb\xfe\x8f\xfe\xa0\xfd\xa9\xfc7\xfc\x12\xfc@\xfco\xfc{\xfbb\xfa=\xf9\xec\xf7\x81\xf7x\xf7\x84\xf6\xe1\xf5\xdc\xf4\xff\xf3;\xf4~\xf4\xa2\xf4!\xf53\xf5\xcd\xf5\x04\xf7O\xf8\xc5\xf9\r\xfb\x1b\xfc\x8a\xfd"\xff\xe0\x00\xbd\x02Y\x04\x9c\x05\x93\x06

In [8]:
sample = next(iter(hf_dataset))

print(type(sample["wav"]))
print(sample["wav"])

if isinstance(sample["wav"], dict):
    print(sample["wav"].keys())

<class 'datasets.features._torchcodec.AudioDecoder'>


In [22]:
# Initialize Model & Optimizer
model = WavLMSpeakerModel(num_speakers=num_speakers).to(DEVICE)
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    [
        model.layer_weights,
        *model.pooling.parameters(),
        *model.aam.parameters()
    ],
    lr=5e-4 # Additive Angular Margin is highly sensitive; keeping it stable
)

# Resuming logic
if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}...")
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    print("Resumed training successfully!")

# Training loop
print("Training started...")
gc.collect()
torch.cuda.empty_cache()

running_loss = 0.0
step = 0
num_steps_to_train = 5000  # Adjust as needed (e.g. 50k steps)

model.train()
stop_training = False

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

Loading checkpoint from /content/drive/MyDrive/best_voxceleb1_wavlm_astp_aam.pth...
Resumed training successfully!
Training started...


In [ ]:
for epoch in range(5):
    if stop_training:
        break
    print(f"\n--- Epoch {epoch+1} ---")
    for input_values, labels in loader:
        input_values = input_values.to(DEVICE)
        labels = labels.to(DEVICE)

        logits, embeddings = model(input_values, labels)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        step += 1

        if step % 100 == 0:
            avg_loss = running_loss / 100
            print(f"Step {step} | Avg Loss (last 100 steps) = {avg_loss:.4f}")
            running_loss = 0.0

        # Save checkpoint periodically
        if step % 5000 == 0:
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Checkpoint saved at step {step} to {checkpoint_path}")

        if step >= num_steps_to_train:
            stop_training = True
            break

# Final Save
torch.save(model.state_dict(), checkpoint_path)
print(f"Training completed! Final model saved to {checkpoint_path}")
print("Final learned layer weights:")
print(torch.softmax(model.layer_weights, dim=0).detach().cpu().numpy())


--- Epoch 1 ---
Step 100 | Avg Loss (last 100 steps) = 12.7718
Step 200 | Avg Loss (last 100 steps) = 9.2778
Step 300 | Avg Loss (last 100 steps) = 15.6348
Step 400 | Avg Loss (last 100 steps) = 14.3525
Step 500 | Avg Loss (last 100 steps) = 14.2269
Step 600 | Avg Loss (last 100 steps) = 15.9065
Step 700 | Avg Loss (last 100 steps) = 15.3453


In [21]:
torch.softmax(model.layer_weights, dim=0)

tensor([0.0780, 0.0803, 0.0815, 0.0815, 0.0803, 0.0794, 0.0779, 0.0725, 0.0722,
        0.0725, 0.0728, 0.0725, 0.0786], device='cuda:0',
       grad_fn=<SoftmaxBackward0>)

In [ ]:
# Install required libraries
!pip install -q transformers datasets soundfile accelerate scikit-learn

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
import os
import gc
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
from datasets import load_dataset
import datasets

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Checkpoint paths
checkpoint_dir = "/content/drive/MyDrive" if os.path.exists("/content/drive/MyDrive") else "/content"
checkpoint_path = os.path.join(checkpoint_dir, "best_voxceleb1_wavlm_astp_aam.pth")

# Split configuration: 1251 total speakers in VoxCeleb1
EVAL_SPEAKERS = set(f"id{i}" for i in range(10001, 10041))        # 40 speakers
TRAIN_SPEAKERS = sorted([f"id{i}" for i in range(10041, 11252)])  # 1211 speakers
spk2idx = {spk: idx for idx, spk in enumerate(TRAIN_SPEAKERS)}
num_speakers = len(TRAIN_SPEAKERS)

def get_speaker_id(key):
    parts = key.split("/")
    if len(parts) >= 3:
        return parts[2]
    return None

#################################################
# ASTP Pooling
#################################################
class ASTP(nn.Module):
    def __init__(self, in_dim, bottleneck_dim=128):
        super().__init__()
        self.linear1 = nn.Conv1d(in_dim, bottleneck_dim, kernel_size=1)
        self.linear2 = nn.Conv1d(bottleneck_dim, in_dim, kernel_size=1)

    def forward(self, x):
        alpha = torch.tanh(self.linear1(x))
        alpha = torch.softmax(self.linear2(alpha), dim=2)
        mean = torch.sum(alpha * x, dim=2)
        var = torch.sum(alpha * (x ** 2), dim=2) - mean ** 2
        std = torch.sqrt(var.clamp(min=1e-7))
        return torch.cat([mean, std], dim=1)

#################################################
# AAM-SoftMax
#################################################
class AAMSoftmax(nn.Module):
    def __init__(self, embedding_dim, num_classes, margin=0.2, scale=30):
        super().__init__()
        self.margin = margin
        self.scale = scale
        self.weight = nn.Parameter(torch.randn(num_classes, embedding_dim))
        nn.init.xavier_normal_(self.weight)

    def forward(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        weight = F.normalize(self.weight, dim=1)
        cosine = F.linear(embeddings, weight)

        theta = torch.acos(torch.clamp(cosine, -1 + 1e-7, 1 - 1e-7))
        target_cosine = torch.cos(theta + self.margin)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1)

        logits = (one_hot * target_cosine + (1 - one_hot) * cosine)
        logits *= self.scale
        return logits

#################################################
# Model
#################################################
class WavLMSpeakerModel(nn.Module):
    def __init__(self, num_speakers):
        super().__init__()
        self.wavlm = WavLMModel.from_pretrained("microsoft/wavlm-base")
        self.layer_weights = nn.Parameter(torch.ones(13))

        for p in self.wavlm.parameters():
            p.requires_grad = False

        self.pooling = ASTP(768)
        self.aam = AAMSoftmax(embedding_dim=1536, num_classes=num_speakers)

    def forward(self, input_values, labels=None):
        with torch.no_grad():
            outputs = self.wavlm(input_values=input_values, output_hidden_states=True)
        hidden_states = outputs.hidden_states

        weights = torch.softmax(self.layer_weights, dim=0)
        combined = 0
        for w, h in zip(weights, hidden_states):
            combined = combined + w * h

        combined = combined.transpose(1, 2)
        embedding = self.pooling(combined)
        embedding = F.normalize(embedding, dim=-1)

        logits = None
        if labels is not None:
            logits = self.aam(embedding, labels)

        return logits, embedding

# Processor and Collate Function
processor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base")

def collate_fn(batch):
    audios = [x[0] for x in batch]
    labels = torch.tensor([x[1] for x in batch])
    features = processor(
        audios,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )
    return features.input_values, labels

#################################################
# Streaming Dataset Wrapper
#################################################
class VoxCelebStreamingDataset(IterableDataset):
    def __init__(self, stream, spk2idx):
        self.stream = stream
        self.spk2idx = spk2idx

    def __iter__(self):
        for sample in self.stream:
            try:
                key = sample["__key__"]
                spk_id = get_speaker_id(key)

                if spk_id not in self.spk2idx:
                    continue

                label = self.spk2idx[spk_id]

                audio_data = sample["wav"]

                print(type(audio_data))
                print(audio_data)

                break

                if len(audio.shape) > 1:
                    audio = audio.squeeze()

                target_len = 48000

                if len(audio) > target_len:
                    start = np.random.randint(
                    0,
                    len(audio) - target_len
                )
                    audio = audio[start:start+target_len]
                else:
                    audio = np.pad(
                    audio,
                    (0, target_len-len(audio))
                )

                yield audio, label

            except Exception as e:
                print("Skipping sample:", e)
                continue

# Load & Filter Dataset
print("Loading Acouspike/Voxceleb1 dataset in streaming mode...")
hf_dataset = load_dataset("Acouspike/Voxceleb1", split="train", streaming=True)
hf_dataset = hf_dataset.cast_column("wav", datasets.Audio(sampling_rate=16000))

# Filter out test speakers from training stream
def is_train_sample(sample):
    spk_id = get_speaker_id(sample["__key__"])
    return spk_id is not None and spk_id not in EVAL_SPEAKERS

train_stream = hf_dataset.filter(is_train_sample)
#train_stream = train_stream.shuffle(buffer_size=1000, seed=42)
dataset = VoxCelebStreamingDataset(train_stream, spk2idx)

loader = DataLoader(
    dataset,
    batch_size=8,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)
print("Testing one batch...")

for input_values, labels in loader:
    print("Input shape:", input_values.shape)
    print("Labels shape:", labels.shape)
    break

print("Batch loaded successfully!")
# Initialize Model & Optimizer
model = WavLMSpeakerModel(num_speakers=num_speakers).to(DEVICE)
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    [
        model.layer_weights,
        *model.pooling.parameters(),
        *model.aam.parameters()
    ],
    lr=5e-4 # Additive Angular Margin is highly sensitive; keeping it stable
)

# Resuming logic
if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}...")
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    print("Resumed training successfully!")

# Training loop
print("Training started...")
gc.collect()
torch.cuda.empty_cache()

running_loss = 0.0
step = 0
num_steps_to_train = 50000  # Adjust as needed (e.g. 50k steps)

model.train()
stop_training = False
for epoch in range(5):
    if stop_training:
        break
    print(f"\n--- Epoch {epoch+1} ---")
    for input_values, labels in loader:
        input_values = input_values.to(DEVICE)
        labels = labels.to(DEVICE)

        logits, embeddings = model(input_values, labels)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        step += 1

        if step % 100 == 0:
            avg_loss = running_loss / 100
            print(f"Step {step} | Avg Loss (last 100 steps) = {avg_loss:.4f}")
            running_loss = 0.0

        # Save checkpoint periodically
        if step % 5000 == 0:
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Checkpoint saved at step {step} to {checkpoint_path}")

        if step >= num_steps_to_train:
            stop_training = True
            break

# Final Save
torch.save(model.state_dict(), checkpoint_path)
print(f"Training completed! Final model saved to {checkpoint_path}")
print("Final learned layer weights:")
print(torch.softmax(model.layer_weights, dim=0).detach().cpu().numpy())

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Loading Acouspike/Voxceleb1 dataset in streaming mode...


README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

Streaming output truncated to the last 5000 lines.
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'
Skipping sample: 'array'

ReadError: unexpected end of data